# ITDAA4 # Student Academic Performance Analysis

This project analyses student academic performance using Python and SQL. The dataset is cleaned and analysed using visualisations, clustering and classification to identify patterns in student performance and tutorial attendance.

In [ ]:
#Azhar Abdool
# ITDAA4
# Data mining and admin assignment
# 01/08/2026

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## Question 1 - Data Preprocessing

First we load the dataset and check the columns, data types, missing values and categorical values before cleaning the data.

In [ ]:
#Azhar Abdool
#Q1
#

df = pd.read_csv('student_performance.csv')

#remove extra index column
df = df.drop(columns=['Unnamed: 0'])

#basic dataset checks
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

print('\nFirst 5 rows:')
display(df.head())

#check column types
print('\nData types:')
display(df.dtypes.to_frame('type'))

#check for missing data
print('\nMissing values:')
display(df.isna().sum().to_frame('missing'))

#check main values before cleaning
for x in ['qualification', 'passed_all_modules', 'study_level', 'gender']:
    print(f'\n{x}:')
    
    temp = df[x].value_counts(dropna=False) #count each unique value
    
    for y, z in temp.items():
        print(y, z)

## 1.1 Standardise qualification and passed_all_modules values
We notice some spelling and capitalisation differences in qualification. The passed_all_modules column also contains ultimately different versions of yes and no, so these values need to be standardised.

In [ ]:
# #save values before cleaning
# q_before = sorted(df['qualification'].dropna().unique())
# p_before = sorted(df['passed_all_modules'].dropna().unique())
# print('Qualification values before cleaning:')
# print(q_before)
# print('\nPassed all modules before cleaning:')
# print(p_before)

#reload original data for comparison
tmp = pd.read_csv('student_performance.csv')#reloading cause After one run the new values were saved as prev vals
tmp = tmp.drop(columns=['Unnamed: 0'])

#values before cleaning
q_before = sorted(tmp['qualification'].dropna().unique())
p_before = sorted(tmp['passed_all_modules'].dropna().unique())
print('Qualification values before cleaning:')
print(q_before)
print('\nPassed all modules before cleaning:')
print(p_before)


#clean qualification values
df['qualification'] = df['qualification'].str.strip()   #remove extra spaces
df['qualification'] = df['qualification'].replace({
    'BSc in CompUter Science': 'BSc in Computer Science',   #fix spelling/case
    'BSC IT: ROBOTICS': 'BSc IT: Robotics'  #standardise case
})

#clean passed values
df['passed_all_modules'] = (
    df['passed_all_modules']
    .str.strip()    #remove extra spaces
    .str.lower()    #make all lowercase first
    .replace({
        'yeah': 'yes',  #convert alternate yes value
        'nope': 'no'    #convert alternate no value
    })
    .str.title()    #final values become Yes / No
)

#check the final cleaned values
print('\nQualification values after cleaning:')
print(sorted(df['qualification'].dropna().unique()))
print('\nPassed all modules after cleaning:')
print(sorted(df['passed_all_modules'].dropna().unique()))

After cleaning, the qualification names are consistent and passed_all_modules contains only Yes and No.

## 1.2 Replace missing tutorial values with zero
The question requires missing values in tutorial_classes_attended and tutorial_class_modules to be replaced with 0. we first check the missing values, replace them, and then confirm that none are left.

In [ ]:
#tutorial columns
tut = ['tutorial_classes_attended', 'tutorial_class_modules']

#check missing values before filling
print('Missing tutorial values before filling:')
display(df[tut].isna().sum().to_frame('before'))

#replace missing values with 0
df[tut] = df[tut].fillna(0)

#check again after filling
print('\nMissing tutorial values after filling:')
display(df[tut].isna().sum().to_frame('after')) #should now be 0

The 4 missing values in each tutorial column were replaced with 0, leaving no missing tutorial values.

## 1.3 Handle remaining missing values
After replacing the missing tutorial values, we check the dataset again for any remaining missing values. For any numeric values that are still missing, we use an appropriate measure to fill them.

In [ ]:
#check what is still missing
print('Missing values before final treatment:')
display(df.isna().sum().to_frame('before'))

#calculate median grade
med = df['grade_average'].median()

#fill missing grades with median
df['grade_average'] = df['grade_average'].fillna(med)
print(f'\nMedian grade used: {med:.1f}')    #not needed but good to know
print('\nMissing values after treatment:')  #final missing value check
display(df.isna().sum().to_frame('after'))

The remaining 3 missing values were in grade_average. I replaced these with the median value of 63.0, which is less affected by unusually high or low marks than the mean.

## Question 2 - Exploratory Data Analysis

The following analysis uses the cleaned student performance data from Question 1.

### 2.1(1a) Academic Performance by Qualification

I grouped the data by qualification and calculated the average pass rate for each one. I also included the number of students in each qualification for context.

In [ ]:
#average pass rate by qualification
q = df.groupby('qualification')['pass_rate'].agg(['count', 'mean']) #group by qualification and get count + average
q.columns = ['students', 'avg_pass']
q = q.sort_values('avg_pass', ascending=False)
display(q.round(2))

#plot the averages-> changed from vertical to horizontal
q['avg_pass'].plot(kind='barh', figsize=(9, 5))

plt.title('Average Pass Rate by Qualification')
plt.xlabel('Average Pass Rate (%)')
plt.ylabel('Qualification')
plt.gca().invert_yaxis()    #to switch it to show highest at the top like the table
plt.tight_layout()
plt.show()

The Higher Certificate in Information Systems: Cyber Security had the highest average pass rate at 89.95%, followed by Software Development at 85.42% and Cloud Computing at 83.02%. BSc IT: Security and Network Engineering had the lowest average pass rate at 56.90%. Overall, the Higher Certificate qualifications performed better than most of the BSc qualifications.

### 2.1(1b) Pass Rate Distributions by Qualification

I use a boxplot to compare the pass rate distributions across qualifications. This shows the median, spread and any possible outliers.

In [ ]:
#compare pass rate distributions
df.boxplot(
    column='pass_rate',
    by='qualification',
    vert=False, #much more readable horizontally
    figsize=(9, 6)
)

plt.title('Pass Rate Distribution by Qualification')
plt.suptitle('')  #remove automatic pandas title
plt.xlabel('Pass Rate (%)')
plt.ylabel('Qualification')
plt.tight_layout()
plt.show()
#summary of pass rates
dist = df.groupby('qualification')['pass_rate'].describe()
display(dist[['count', 'mean', '50%', 'std', 'min', 'max']].round(2))

The Higher Certificate qualifications generally have the strongest pass rate distributions. Cyber Security and Software Development both have a median pass rate of 100%, while Cloud Computing has a median of 90.3%. BSc IT: Security and Network Engineering performs noticeably lower, with a median of 64.3% and a mean of 56.9%. The spread also differs across qualifications, with Software Engineering showing the largest standard deviation at 33.25, indicating more variation in student performance. Several qualifications also have very low minimun values and visible low end outliers, showing that although many students perform well, some perform much worse than the typical student in their qualification.

### 2.1 (2a) Tutorial Attendance and Passing All Modules

Comparing the percentage who passed all modules for students who attended tutorials vs those who did not.
(students who attended at least one tutorial with students who did not attend any tutorials and calculate the percentage who passed all modules)

In [ ]:
#mark whether a student attended tutorials
df['tut'] = df['tutorial_classes_attended'] > 0
#count students and passes in each group
t = df.groupby('tut')['passed_all_modules'].agg(
    students='size',
    passed=lambda x: (x == 'Yes').sum()
)

#calculate pass percentage
t['pass_pct'] = t['passed'] / t['students'] * 100
display(t.round(2))

#plot pass percentage
p = t.reindex([False, True])['pass_pct']
p.plot(
    kind='bar',
    figsize=(7, 4),
    color=['red', 'green']
)
plt.title('Passed All Modules by Tutorial Attendance')
plt.xlabel('Tutorial Attendance')
plt.ylabel('Passed All Modules (%)')
plt.xticks([0, 1], ['No Tutorials', 'Attended Tutorials'], rotation=0)
plt.tight_layout()
plt.show()

Students who did not attend tutorials had a higher pass all modules rate of 63.76%, compared with 50.13% for students who attended tutorials. This shows a negative association between tutorial attendance and passing all modules at the overall faculty level. However, this does not mean tutorials cause lower performance, as students who are already struggling may be more likely to attend tutorials.

### 2.1 (2b) Tutorial Attendance by Qualification

Comparing tutorial attendees and non attendees within each qualification to see whether the overall pattern is the same across the faculty.

In [ ]:
#pass rates by qualification and tutorial attendance
qt = (
    df.groupby(['qualification', 'tut'])['passed_all_modules']
    .agg(
        students='size',
        passed=lambda x: (x == 'Yes').sum()
    )
    .reset_index()
)

#calculate percentage passed
qt['pass_pct'] = qt['passed'] / qt['students'] * 100
display(qt.round(2))

#reshape for graph
qp = qt.pivot(
    index='qualification',
    columns='tut',
    values='pass_pct'
)
qp.columns = ['No Tutorials', 'Attended Tutorials']
#keep the same qualification order as 2.1(a)
qp = qp.reindex(q.index)

#plot comparison
ax = qp.plot(
    kind='barh',
    figsize=(10, 6),
    color=['red', 'green']
)
plt.title('Passed All Modules by Qualification and Tutorial Attendance')
plt.xlabel('Passed All Modules (%)')
plt.ylabel('Qualification')

#highest performing qualifications at the top
plt.gca().invert_yaxis()

#move legend outside graph
plt.legend(
    title='Tutorial Attendance',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

#show percentages on bars
for x in ax.containers:
    ax.bar_label(x, fmt='%.1f%%', padding=3, fontsize=8)
plt.xlim(0, 100)    #because otherwise limits
plt.tight_layout()
plt.show()

The relationship between tutorial attendance and passing all modules is not consistent across qualifications. In most qualifications, students who did not attend tutorials had a higher pass all modules percentage. Software Development was the main exception, where tutorial attendees had a slightly higher pass rate (61.55%) than non attendees (58.33%). However, some groups contain very few students, such as Cyber Security with only one tutorial attendee, so these comparisons should not be treated as strong evidence of tutorial effectiveness.


### 2.1 (2c) Attendance by Study Level

Calculatind the average number of tutorial classes attended for each study level to compare tutorial engagement across the different years.

In [ ]:
#study level order
lvl = ['1st year', '2nd year', '3rd year']

#average tutorial attendance by level
s = df.groupby('study_level')['tutorial_classes_attended'].mean()
s = s.reindex(lvl)
display(s.round(2).to_frame('avg_tutorials'))

#plot averages
ax = s.plot(kind='bar', figsize=(7, 4))
plt.title('Average Tutorial Classes Attended by Study Level')
plt.xlabel('Study Level')
plt.ylabel('Average Tutorial Classes Attended')
plt.xticks(rotation=0)

#addin values above bars
for x, y in enumerate(s):
    ax.text(x, y + 0.5, f'{y:.1f}', ha='center')
plt.tight_layout()
plt.show()

First year students had the highest average tutorial attendance at 30.19 classes, followed closely by second year students at 29.20. Third year students attended far fewer tutorials on average, at only 7.42 classes. This suggests tut attendence is not only not consistent across years but decreases. 

## 2.2 Findings and Recommendations

The highest average pass rate was for Higher Certificate in Information Systems: Cyber Security at 89.95%, while BSc IT: Security and Network Engineering had the lowest at 56.90%. The pass rate distributions also differed between qualifications, with Software Engineering showing the greatest variation in results. Overall, students who attended tutorials had a lower pass all modules rate of 50.13% compared with 63.76% for students who did not attend. This pattern was also seen in most qualifications, although Software Development was an exception. Tutorial attendance was highest among 1st year students and lowest among 3rd year students.

Recommendations:

1.Give more academic support to BSc IT: Security and Network Engineering because it had the lowest average pass rate.
2.Review how tutorials are being used before trying to increase attendance, since tutorial attendees had lower pass rates overall.
3.Use different support plans for each qualification instead of one general approach for the whole faculty.
4.Identify students with low pass rates or low grade averages early so support can be offered before final results.
5.Investigate why 3rd year tutorial attendance is much lower than 1st and 2nd year. Possibly exploring more flexible times or online options for busy final years. 
6.Look at what is working well in the stronger qualifications and see whether those approaches can be used in weaker programmes.


## Question 3 - K-Means Clustering

### 3(a) Feature Selection

selecting numeric features that represent student performance, workload and tutorial engagement. I also converted study_level to a numeric value so it could be included in the clustering.

In [ ]:
#convert study level to numbers
lvl_map = {
    '1st year': 1,
    '2nd year': 2,
    '3rd year': 3
}
df['lvl_num'] = df['study_level'].map(lvl_map)  #convert study level text to numbers ->1st=1, 2nd=2, 3rd=3

#features used for clustering
f = [
    'modules',
    'module_credits',
    'grade_average',
    'modules_passed',
    'pass_rate',
    'tutorial_classes_attended',
    'tutorial_class_modules',
    'lvl_num'
]
print('Clustering features:')
print(f)
display(df[f].describe().round(2))

These features cover academic performance, workload, progression and tutorial engagement, which are the main areas required for the student segmentation.

### 3(b) Feature Scaling

In [ ]:
#select clustering features
x = df[f]

#scale the data
sc = StandardScaler()
xs = sc.fit_transform(x)
print('Scaled data shape:', xs.shape)

#check scaled values
display(pd.DataFrame(xs, columns=f).describe().round(2))

Scaling is needed because K-means is based on distance. Without scaling, features with larger values could have more influence on the clusters than the others.

### 3(c) PCA

In [ ]:
#reduce to 2 principal components
pc = PCA(n_components=2)
p = pc.fit_transform(xs)

#save PCA values
df['PC1'] = p[:, 0]
df['PC2'] = p[:, 1]
print('Explained variance:', pc.explained_variance_ratio_)
print('Total variance explained:', pc.explained_variance_ratio_.sum())
#makin it a little cleaner
print('PC1 variance:', round(pc.explained_variance_ratio_[0] * 100, 1), '%')
print('PC2 variance:', round(pc.explained_variance_ratio_[1] * 100, 1), '%')
print('Total variance:', round(pc.explained_variance_ratio_.sum() * 100, 1), '%')

The first two principal components explain about 72.7% of the variation in the scaled data. This keeps most of the information while allowing the clusters to be shown on a 2D graph.

### 3(d) Determine the Optimal Number of Clusters

In [ ]:
#test different k values
res = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(xs)
    res.append([
        k,
        km.inertia_,
        silhouette_score(xs, lab)
    ])

#convert results to dataframe
res = pd.DataFrame(res, columns=['k', 'inertia', 'silhouette'])
display(res.round(4))

#elbow method
plt.figure(figsize=(6, 4))
plt.plot(res['k'], res['inertia'], marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.show()

#silhouette scores
plt.figure(figsize=(6, 4))
plt.plot(res['k'], res['silhouette'], marker='o')
plt.title('Silhouette Score by k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.show()

The silhouette score is highest at k = 2 with a score of 0.375. The elbow plot also shows that the biggest improvements happen at the lower k values before the curve starts to flatten. I therefore use k = 2 for the clustering.

### 3(e) Fitting the K-means model

Based on the elbow and silhouette results, I use 2 clusters and fit the K-means model to the scaled data.

In [ ]:
#fit final k-means model ----2
km = KMeans(n_clusters=2, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(xs)

#average values for each cluster
prof = df.groupby('cluster')[f].mean().round(2)

#add number of students in each cluster
size = df['cluster'].value_counts().sort_index()
prof.insert(0, 'students', size)
display(prof)

Cluster 0 contains 856 students and shows stronger academic performance, with an average pass rate of 91.26% and higher tutorial attendance of 34.88 classes. Cluster 1 contains 492 students and has lower academic performance, with an averag pass rate of 67.41% and lower tut attendance of 8.27 classes. Cluster 0 also mainly contains earlier year students, while Cluster 1 contains students at higher study levels on average.

### 3(f) Cluster Visualisation

In [ ]:
#move cluster centroids into PCA space
cent = pc.transform(km.cluster_centers_)

#colours for clusters and markers for study level
cols = ['red', 'gray']
marks = {
    '1st year': 'o',
    '2nd year': 's',
    '3rd year': '^'
}
fig, ax = plt.subplots(figsize=(10, 6))

#plot each cluster and study level
for c in range(2):
    for lvl, m in marks.items():
        mask = (df['cluster'] == c) & (df['study_level'] == lvl)
        ax.scatter(
            df.loc[mask, 'PC1'],
            df.loc[mask, 'PC2'],
            color=cols[c],
            marker=m,
            alpha=0.6,
            s=35
        )
#plot centroids
ax.scatter(
    cent[:, 0],
    cent[:, 1],
    color='black',
    marker='X',
    s=180
)

#label centroids
for i, (x, y) in enumerate(cent):
    ax.text(x + 0.1, y, str(i), fontsize=11, weight='bold')
plt.title('K-Means Clusters in PCA Space')
plt.xlabel('PC1')
plt.ylabel('PC2')

#legend
cluster_legend = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor='red', label='Cluster 0', markersize=8),

    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor='gray', label='Cluster 1', markersize=8),

    plt.Line2D([0], [0], marker='o', color='gray',
               linestyle='None', label='1st year'),

    plt.Line2D([0], [0], marker='s', color='gray',
               linestyle='None', label='2nd year'),

    plt.Line2D([0], [0], marker='^', color='gray',
               linestyle='None', label='3rd year'),

    plt.Line2D([0], [0], marker='X', color='black',
               linestyle='None', label='Centroid')
]
ax.legend(handles=cluster_legend, loc='best')
plt.tight_layout()
plt.show()

The PCA plot shows a clear separation between the two clusters, mainly along PC1. Cluster 0 contains more of the higher-performing and earlier year students, while Cluster 1 contains more students from later study levels. The black X markers show the clusters centroids.

## Question 4 - Classification

Loading the AI student performance dataset and inspecting the columns, data types, target values and missing data before building the models.

In [ ]:
ai = pd.read_csv('ai_impact_student_performance_dataset.csv')

#basic checks
print('Shape:', ai.shape)
print('\nColumns:')
print(ai.columns.tolist())
print('\nData types:')
display(ai.dtypes.to_frame('type'))

#target distribution
print('\nPassed values:')
display(ai['passed'].value_counts().to_frame('count'))

#missing values
print('\nMissing values:')
display(ai.isna().sum().to_frame('missing'))

#split column types
cat = ai.drop(columns='passed').select_dtypes(include=['object', 'string']).columns.tolist()
num = ai.drop(columns='passed').select_dtypes(include='number').columns.tolist()
print('\nCategorical columns:')
print(cat)
print('\nNumeric columns:')
print(num)

The provided dataset contains 8,000 student records rather than the 1,000 stated in the scenario. Missing values are only present in ai_tools_used and ai_usage_purpose, and most students in the dataset passed. I used the full dataset provided for the modelling.

### 4.1(a and b) Missing Values and Categorical Data

Before training the models, Ill remove columns that should not be used for prediction and separate the numeric and categorical features. student_id is removed because it is only an identifier, while final_score and performance_category summarise the final performance outcome and could cause target leakage.

In [ ]:
#target and columns not used for prediction
y = ai['passed']
drop = ['student_id', 'final_score', 'performance_category']
x = ai.drop(columns=['passed'] + drop)
print('Removed columns:', drop)

#split numeric and categorical columns
cat = x.select_dtypes(include=['object', 'string']).columns.tolist()
num = x.select_dtypes(include='number').columns.tolist()



The missing values are in the categorical ai_tools_used and ai_usage_purpose columns. I use the most common value to fill missing categorical data. Numeric columns are also set to use median imputation if any missing values are encountered during modelling.

Categorical variables are one hot encoded so they can be used by the classification models. handle_unknown='ignore' is used so that unseen categories in the test data do not cause errors.

### 4.1(c) Train/Test Split

I split the data into 80% training and 20% testing data. Stratification is used because the dataset contains many more passed students than failed students.

In [ ]:

#80/20 train-test split
xtr, xte, ytr, yte = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print('Training shape:', xtr.shape)
print('Testing shape:', xte.shape)

#check training target balance
display(
    ytr.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame('percent')
)



The training data remains imbalanced, with about 88.9% passed and 11.1% failed, so accuracy alone will not be enough when comparing the models.

### 4.1(d) Feature Scaling

Ill create separate preprocessing steps for the two models. Logistic Regression uses median imputation for numeric values, most-frequent imputation and one-hot encoding for categorical values, and scaling for numeric features. The Decision Tree uses the same missing-value and categorical treatment but does not need numeric scaling.

In [ ]:
#preprocessing for logistic regression
log_pre = ColumnTransformer([
    ('num', Pipeline([
        ('fill', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num),
    ('cat', Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat)
])

#preprocessing for decision tree
tree_pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num),

    ('cat', Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat)
])

### 4.1(e) Feature Selection

Using SelectKBest to keep the strongest predictive features from the training data after preprocessing. This is done inside the modelling pipeline so that feature selection does not use information from the test data.

In [ ]:
k=15    #I keep the 15 highest scoring features to reduce the number of predictors while still retaining a useful range of academic, behavioural and AI-related variables.
#logistic regression pipeline
log= Pipeline([
    ('pre', log_pre),
    ('select', SelectKBest(f_classif, k=k)),
    ('model', LogisticRegression(max_iter=1000))
])
#decision tree pipeline
tree= Pipeline([
    ('pre', tree_pre),
    ('select', SelectKBest(f_classif, k=k)),
    ('model', DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])
#fit logistic model and view selected features
log.fit(xtr, ytr)

names= log.named_steps['pre'].get_feature_names_out()
mask= log.named_steps['select'].get_support()

sel= names[mask]
sel = [z.replace('num__', '').replace('cat__', '') for z in sel]
print('Selected features:')
print(sel)


### 4.1(f and g) Models and Evaluation

In [ ]:
#fit the decision tree
tree.fit(xtr, ytr)

#predictions
log_pred = log.predict(xte)
tree_pred = tree.predict(xte)

#model results
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(yte, log_pred),
        accuracy_score(yte, tree_pred)
    ],
    'Precision': [
        precision_score(yte, log_pred),
        precision_score(yte, tree_pred)
    ],
    'Recall': [
        recall_score(yte, log_pred),
        recall_score(yte, tree_pred)
    ],
    'F1': [
        f1_score(yte, log_pred),
        f1_score(yte, tree_pred)
    ]
})
display(results.round(3))

#logistic regression confusion matrix
cm_log = confusion_matrix(yte, log_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm_log, cmap='Blues')
plt.title('Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0, 1], ['Not Passed', 'Passed'])
plt.yticks([0, 1], ['Not Passed', 'Passed'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_log[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

#decision tree confusion matrix
cm_tree = confusion_matrix(yte, tree_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm_tree, cmap='Blues')
plt.title('Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0, 1], ['Not Passed', 'Passed'])
plt.yticks([0, 1], ['Not Passed', 'Passed'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_tree[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

Logistic Regression performed better overall, with higher accuracy, precision, recall and F1 score than the Decision Tree. The confusion matrices also show that Logistic rwegression made fewer incorrect predictions. Because the dataset is imbalanced, the F1 score is especially useful and also supports Logistic Regression as the better model.

## 4.2 Model Comparison

Logistic Regression performed best on the test data, with an F1 score of 0.971 compared with 0.960 for the Decision Tree. It also had higher accuracy, precision and recall. Logistic Regression is simpler and easier to explain, although it may not capture complex nonlinear patterns as well as a decision tree. The decision tree can model more complex relationships, but it can also overfit, which is why I limited the tree depth. Because the dataset is imbalanced, accuracy should not be considered on its own, so I also compare precision, recall, F1 and the confusion matrices.

## Question 5 - SQL

The following SQL queries use the Student_Performance table and fields provided in the question.
### 5.1 The total number of student records in the database.

```sql
SELECT COUNT(*) AS total_student_records
FROM Student_Performance;
```

### 5.2 The average pass rate across all students

```sql
SELECT AVG(pass_rate) AS average_pass_rate
FROM Student_Performance;
```

### 5.3 The number of students who passed all modules.

```sql
SELECT COUNT(*) AS passed_all_modules_count
FROM Student_Performance
WHERE LOWER(passed_all_modules) = 'yes';
```

### 5.4 The average tutorial classes attended per study level.

```sql
SELECT
    study_level,
    AVG(tutorial_classes_attended) AS avg_tutorial_classes_attended
FROM Student_Performance
GROUP BY study_level
ORDER BY study_level;
```

### 5.5 The qualification with the highest average pass rate.

```sql
SELECT
    qualification,
    AVG(pass_rate) AS average_pass_rate
FROM Student_Performance
GROUP BY qualification
ORDER BY average_pass_rate DESC
FETCH FIRST 1 ROW ONLY;
```